# F1 Lap Time Prediction — XGBoost Model

**Circuit:** Bahrain Grand Prix (2022–2025)  
**Model:** XGBoost Regressor  
**Inputs:** Driver, LapNumber, Compound, TyreLife, Year  
**Target:** LapTime (seconds)  

Only drivers who maintained the same team across all 4 years are included:  
ALB (Williams), LEC (Ferrari), NOR (McLaren), RUS (Mercedes), VER (Red Bull)

### Year-over-Year Trend
A per-driver yearly improvement rate is computed so the model can extrapolate predictions to future years (e.g. 2026).

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")

In [2]:
files = sorted(glob.glob('datasets/Bahrain/*_Laps.csv'))
df_list = []

for f in files:
    temp_df = pd.read_csv(f)
    
    year = int(os.path.basename(f).split('_')[0])
    temp_df['Year'] = year
    df_list.append(temp_df)
    print(f"Loaded {len(temp_df)} laps from {f}")

df = pd.concat(df_list, ignore_index=True)
print(f"\nTotal laps loaded: {len(df)}")
df.head()

Loaded 1125 laps from datasets/Bahrain\2022_Bahrain_Laps.csv
Loaded 1056 laps from datasets/Bahrain\2023_Bahrain_Laps.csv
Loaded 1129 laps from datasets/Bahrain\2024_Bahrain_Laps.csv
Loaded 1128 laps from datasets/Bahrain\2025_Bahrain_Laps.csv

Total laps loaded: 4438


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year
0,0 days 01:04:15.340000,VER,1,0 days 00:01:40.236000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:42.325000,...,Red Bull Racing,0 days 01:02:34.872000,NaT,1,2.0,NaN,NaN,False,False,2022
1,0 days 01:05:53.220000,VER,1,0 days 00:01:37.880000,2.0,1.0,NaT,NaT,0 days 00:00:31.285000,0 days 00:00:42.269000,...,Red Bull Racing,0 days 01:04:15.340000,NaT,1,2.0,NaN,NaN,False,True,2022
2,0 days 01:07:31.577000,VER,1,0 days 00:01:38.357000,3.0,1.0,NaT,NaT,0 days 00:00:31.499000,0 days 00:00:42.474000,...,Red Bull Racing,0 days 01:05:53.220000,NaT,1,2.0,NaN,NaN,False,True,2022
3,0 days 01:09:10.143000,VER,1,0 days 00:01:38.566000,4.0,1.0,NaT,NaT,0 days 00:00:31.342000,0 days 00:00:42.674000,...,Red Bull Racing,0 days 01:07:31.577000,NaT,1,2.0,NaN,NaN,False,True,2022
4,0 days 01:10:49.020000,VER,1,0 days 00:01:38.877000,5.0,1.0,NaT,NaT,0 days 00:00:31.498000,0 days 00:00:42.854000,...,Red Bull Racing,0 days 01:09:10.143000,NaT,1,2.0,NaN,NaN,False,True,2022


In [3]:
df = df.replace("NaT", np.nan)

ELIGIBLE_DRIVERS = ['ALB', 'LEC', 'NOR', 'RUS', 'VER']

print(f"Starting rows: {len(df)}")

# Keep only eligible drivers
df = df[df['Driver'].isin(ELIGIBLE_DRIVERS)].copy()
print(f"After filtering eligible drivers: {len(df)}")

# Convert LapTime from timedelta string to seconds
df['LapTime'] = pd.to_timedelta(df['LapTime']).dt.total_seconds()

# Drop rows where LapTime is NaN
df = df.dropna(subset=['LapTime'])
print(f"After dropping NaN LapTime: {len(df)}")

# Keep only accurate laps
df = df[df['IsAccurate'] == True]
print(f"After keeping IsAccurate only: {len(df)}")

# Remove pit-out laps (PitOutTime is not NaN)
df = df[df['PitOutTime'].isna()]
print(f"After removing pit-out laps: {len(df)}")

# Remove pit-in laps (PitInTime is not NaN)
df = df[df['PitInTime'].isna()]
print(f"After removing pit-in laps: {len(df)}")

# Remove first lap
df = df[df['LapNumber'] > 1]
print(f"After removing first laps: {len(df)}")

# Green flag only
df = df[df['TrackStatus'] == 1]
print(f"After green-flag filter: {len(df)}")

print(f"\n✅ Final clean dataset: {len(df)} laps")
print(f"Drivers: {sorted(df['Driver'].unique())}")
print(f"Years: {sorted(df['Year'].unique())}")
print(f"Compounds: {sorted(df['Compound'].unique())}")
df[['Driver', 'LapNumber', 'Compound', 'TyreLife', 'LapTime', 'Year']].head(10)

Starting rows: 4438
After filtering eligible drivers: 1117
After dropping NaN LapTime: 1108
After keeping IsAccurate only: 936
After removing pit-out laps: 936
After removing pit-in laps: 936
After removing first laps: 936
After green-flag filter: 916

✅ Final clean dataset: 916 laps
Drivers: ['ALB', 'LEC', 'NOR', 'RUS', 'VER']
Years: [2022, 2023, 2024, 2025]
Compounds: ['HARD', 'MEDIUM', 'SOFT']


C:\Users\roshi\AppData\Local\Temp\ipykernel_15952\486328011.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("NaT", np.nan)


,Driver,LapNumber,Compound,TyreLife,LapTime,Year
1,VER,2.0,SOFT,5.0,97.880,2022
2,VER,3.0,SOFT,6.0,98.357,2022
3,VER,4.0,SOFT,7.0,98.566,2022
4,VER,5.0,SOFT,8.0,98.877,2022
5,VER,6.0,SOFT,9.0,98.940,2022
6,VER,7.0,SOFT,10.0,99.062,2022
7,VER,8.0,SOFT,11.0,99.092,2022
8,VER,9.0,SOFT,12.0,99.251,2022
9,VER,10.0,SOFT,13.0,99.392,2022
10,VER,11.0,SOFT,14.0,99.479,2022


In [4]:
yearly_avg = df.groupby(['Driver', 'Year'])['LapTime'].mean().reset_index()
yearly_avg.columns = ['Driver', 'Year', 'AvgLapTime']

from scipy.stats import linregress

driver_trends = {}
print("Per-Driver Yearly Improvement (linear trend):")
print("=" * 65)

# Per driver improvement
for driver in sorted(ELIGIBLE_DRIVERS):
    d = yearly_avg[yearly_avg['Driver'] == driver]
    slope, intercept, _, _, _ = linregress(d['Year'], d['AvgLapTime'])
    driver_trends[driver] = {'slope': slope, 'intercept': intercept}
    direction = 'faster' if slope < 0 else 'slower'
    print(f"  {driver}: {slope:+.4f} s/year ({abs(slope):.2f}s {direction} each year)")

# Overall average improvement
overall_avg = df.groupby('Year')['LapTime'].mean()
overall_slope, overall_intercept, _, _, _ = linregress(overall_avg.index, overall_avg.values)
print(f"\n  Overall: {overall_slope:+.4f} s/year")


Per-Driver Yearly Improvement (linear trend):
  ALB: -0.6904 s/year (0.69s faster each year)
  LEC: -0.1771 s/year (0.18s faster each year)
  NOR: -1.1631 s/year (1.16s faster each year)
  RUS: -0.3562 s/year (0.36s faster each year)
  VER: -0.1147 s/year (0.11s faster each year)

  Overall: -0.4951 s/year


In [5]:
# Label-encode Driver
le_driver = LabelEncoder()
df['Driver_encoded'] = le_driver.fit_transform(df['Driver'])

print("Driver encoding:")
for cls, lbl in zip(le_driver.classes_, range(len(le_driver.classes_))):
    print(f"  {cls} => {lbl}")

# One-hot encode Compound
compound_dummies = pd.get_dummies(df['Compound'], prefix='Compound')
df = pd.concat([df, compound_dummies], axis=1)

# Define feature columns and target — Year is now included
FEATURE_COLS = ['Driver_encoded', 'LapNumber', 'TyreLife', 'Year'] + \
               [c for c in df.columns if c.startswith('Compound_')]

TARGET_COL = 'LapTime'

print(f"\nFeatures: {FEATURE_COLS}")
print(f"Target: {TARGET_COL}")
print(f"\nFeature matrix shape: {df[FEATURE_COLS].shape}")
df[FEATURE_COLS + [TARGET_COL]].describe()

Driver encoding:
  ALB => 0
  LEC => 1
  NOR => 2
  RUS => 3
  VER => 4

Features: ['Driver_encoded', 'LapNumber', 'TyreLife', 'Year', 'Compound_HARD', 'Compound_MEDIUM', 'Compound_SOFT']
Target: LapTime

Feature matrix shape: (916, 7)


,Driver_encoded,LapNumber,TyreLife,Year,LapTime
count,916.000000,916.000000,916.000000,916.000000,916.000000
mean,1.984716,28.320961,10.587336,2023.538210,97.886221
std,1.431032,16.091869,5.653570,1.104221,1.655531
min,0.000000,2.000000,2.000000,2022.000000,92.608000
25%,1.000000,15.000000,6.000000,2023.000000,96.886500
50%,2.000000,27.000000,10.000000,2024.000000,97.813500
75%,3.000000,42.000000,14.000000,2024.000000,98.773000
max,4.000000,57.000000,31.000000,2025.000000,103.500000


In [6]:
X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['Driver']
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

Training set: 732 samples
Test set:     184 samples


In [7]:
model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42,
    objective='reg:squarederror'
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

print("Model trained successfully!")
print(f"Estimators: {model.n_estimators}")
print(f"Max depth:  {model.max_depth}")
print(f"Features:   {FEATURE_COLS}")

Model trained successfully!
Estimators: 200
Max depth:  6
Features:   ['Driver_encoded', 'LapNumber', 'TyreLife', 'Year', 'Compound_HARD', 'Compound_MEDIUM', 'Compound_SOFT']


In [8]:
y_pred = model.predict(X_test)

# Overall metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("="*50)
print("  Overall Test Set Metrics")
print("="*50)
print(f"  RMSE : {rmse:.4f} s")
print(f"  MAE  : {mae:.4f} s")
print(f"  R²   : {r2:.4f}")
print("="*50)

# Per-driver metrics
test_df = X_test.copy()
test_df['y_true'] = y_test.values
test_df['y_pred'] = y_pred

print("\n  Per-Driver Metrics")
print("-"*60)
print(f"  {'Driver':<8} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'Samples':>8}")
print("-"*60)

for driver_code in sorted(test_df['Driver_encoded'].unique()):
    driver_name = le_driver.inverse_transform([int(driver_code)])[0]
    mask = test_df['Driver_encoded'] == driver_code
    yt = test_df.loc[mask, 'y_true']
    yp = test_df.loc[mask, 'y_pred']
    d_rmse = np.sqrt(mean_squared_error(yt, yp))
    d_mae  = mean_absolute_error(yt, yp)
    d_r2   = r2_score(yt, yp) if len(yt) > 1 else float('nan')
    print(f"  {driver_name:<8} {d_rmse:>8.4f} {d_mae:>8.4f} {d_r2:>8.4f} {len(yt):>8}")

  Overall Test Set Metrics
  RMSE : 0.4948 s
  MAE  : 0.3288 s
  R²   : 0.9045

  Per-Driver Metrics
------------------------------------------------------------
  Driver       RMSE      MAE       R²  Samples
------------------------------------------------------------
  ALB        0.3351   0.2806   0.9315       39
  LEC        0.4458   0.3187   0.8838       36
  NOR        0.7001   0.4587   0.9005       36
  RUS        0.3293   0.2219   0.9458       35
  VER        0.5639   0.3634   0.7522       38


In [9]:

def predict_future_year(model, driver, lap_number, compound, tyre_life, target_year,
                        le_driver=le_driver, driver_trends=driver_trends,
                        feature_cols=FEATURE_COLS, max_train_year=2025):
    """
    Predict lap time for a future year by combining:
      - XGBoost prediction at Year=max_train_year
      - Linear yearly improvement adjustment
    """
    driver_enc = le_driver.transform([driver])[0]
    
    # Build feature dict with Year = max training year
    features = {
        'Driver_encoded': driver_enc,
        'LapNumber': lap_number,
        'TyreLife': tyre_life,
        'Year': max_train_year,
    }
    # One-hot compound columns
    for col in feature_cols:
        if col.startswith('Compound_'):
            features[col] = 1 if col == f'Compound_{compound}' else 0
    
    X_input = pd.DataFrame([features])[feature_cols]
    base_pred = model.predict(X_input)[0]
    
    # Apply yearly improvement for the extra years
    years_ahead = target_year - max_train_year
    yearly_improvement = driver_trends[driver]['slope']  # negative = getting faster
    adjusted_pred = base_pred + (yearly_improvement * years_ahead)
    
    return adjusted_pred, base_pred, yearly_improvement


# --- Demo: Predict lap times for 2026 ---
print("Predicted lap times for 2026 Bahrain GP (mid-race, Lap 30, Medium tyres, TyreLife 10)")
print("=" * 75)
print(f"  {'Driver':<8} {'2025 Pred':>12} {'Yearly Δ':>12} {'2026 Pred':>12}")
print("-" * 75)

for driver in sorted(ELIGIBLE_DRIVERS):
    pred_2026, pred_2025, yearly_delta = predict_future_year(
        model, driver, lap_number=30, compound='MEDIUM', tyre_life=10, target_year=2026
    )
    print(f"  {driver:<8} {pred_2025:>10.3f} s {yearly_delta:>+10.3f} s {pred_2026:>10.3f} s")

Predicted lap times for 2026 Bahrain GP (mid-race, Lap 30, Medium tyres, TyreLife 10)
  Driver      2025 Pred     Yearly Δ    2026 Pred
---------------------------------------------------------------------------
  ALB          99.601 s     -0.690 s     98.910 s
  LEC          97.818 s     -0.177 s     97.641 s
  NOR          97.721 s     -1.163 s     96.558 s
  RUS          97.813 s     -0.356 s     97.456 s
  VER          98.610 s     -0.115 s     98.496 s
